In [1]:
import sys
import os
from pathlib import Path

# Manually add the project root to python path so we can import our new module
# This simulates running from the root C:\streaming_emulator
project_root = Path("C:/streaming_emulator")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Project Root added: {project_root}")

Project Root added: C:\streaming_emulator


In [2]:
from DTC.src import config

print(f"Base Directory: {config.BASE_DIR}")
print(f"Contracts Dir:  {config.CONTRACTS_DIR}")
print(f"Data Directory: {config.DATA_DIR}")

# Check if crucial paths actually exist
print(f"\n[Check] Master JSON Exists? {config.DTC_MASTER_PATH.exists()}")
print(f"[Check] Data Dir Exists?    {config.DATA_DIR.exists()}")
     

Base Directory: C:\streaming_emulator
Contracts Dir:  C:\streaming_emulator\contracts
Data Directory: C:\streaming_emulator\data\vehicles

[Check] Master JSON Exists? True
[Check] Data Dir Exists?    True


In [3]:
# Try to load the config for a specific DTC (P0217 - Engine Overheat)
try:
    dtc_config = config.get_dtc_config("engine", "P0217")
    print("\n[Success] Loaded Config for P0217:")
    print(f" - Description: {dtc_config['description']}")
    print(f" - Criticality: {dtc_config['severity']}")
    print(f" - Features:    {dtc_config['features']}")
except Exception as e:
    print(f"\n[Error] Failed to load DTC config: {e}")


[Success] Loaded Config for P0217:
 - Description: Engine Coolant Over Temperature
 - Criticality: critical
 - Features:    ['ecu_7ea_engine_coolant_temperature', 'engine_oil_temperature', 'engine_load_absolute', 'ecu_7eb_engine_rpm_rpm']


In [4]:
synth_path, artifact_path = config.ensure_dirs("engine", "P0217")

print(f"\n[Check] Synth Path:    {synth_path}")
print(f"[Check] Artifact Path: {artifact_path}")
print(f"Created? {artifact_path.exists()}")


[Check] Synth Path:    C:\streaming_emulator\DTC\synth_data\engine
[Check] Artifact Path: C:\streaming_emulator\DTC\artifacts\engine\P0217
Created? True


In [5]:
import sys
import pandas as pd
from pathlib import Path

# Fix path to import local modules
project_root = Path("C:/streaming_emulator")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from DTC.src.fault_injector import FaultInjector

print("Imports successful. FaultInjector loaded.")

Imports successful. FaultInjector loaded.


In [6]:
# Initialize for Engine module
# This should print "[ENGINE] Loading reference data source: ..."
injector = FaultInjector(module_name="engine")

print(f"Injector ready. Loaded {len(injector.raw_data)} healthy rows.")
print("Columns available:", list(injector.raw_data.columns[:5]), "...")
     

[ENGINE] Loading reference data source: synthetic_engine_inference_scenarioA_sim001.csv
Injector ready. Loaded 756000 healthy rows.
Columns available: ['timestamp', 'date', 'source_id', 'air_fuel_ratio_commanded_1', 'air_fuel_ratio_measured_1'] ...


In [7]:
# P0217 = Engine Coolant Over Temperature
# We expect the 'ecu_7ea_engine_coolant_temperature' to be higher in the faulty section.

dtc_code = "P0217"
df_p0217 = injector.create_dataset(dtc_code, n_samples=1000)

print(f"\nGenerated Dataset for {dtc_code}")
print(f"Total Rows: {len(df_p0217)}")
print(f"Target Distribution:\n{df_p0217['target'].value_counts(bins=2)}")

   -> Generating 1000 samples for P0217 using features: ['ecu_7ea_engine_coolant_temperature', 'engine_oil_temperature', 'engine_load_absolute', 'ecu_7eb_engine_rpm_rpm']

Generated Dataset for P0217
Total Rows: 1000
Target Distribution:
(-0.002, 0.5]    722
(0.5, 1.0]       278
Name: count, dtype: int64


In [8]:
# Verify that the physics logic actually changed the data
# We compare the mean temperature of Healthy rows (target=0) vs Critical rows (target > 0.8)

feature_name = 'ecu_7ea_engine_coolant_temperature'

mean_healthy = df_p0217[df_p0217['target'] == 0.0][feature_name].mean()
mean_critical = df_p0217[df_p0217['target'] > 0.9][feature_name].mean()

print(f"--- Logic Verification for {dtc_code} ---")
print(f"Feature: {feature_name}")
print(f"Avg Healthy Value:  {mean_healthy:.2f}")
print(f"Avg Critical Value: {mean_critical:.2f}")

if mean_critical > mean_healthy:
    print("\n✅ SUCCESS: Critical temperature is higher than healthy temperature.")
else:
    print("\n❌ FAILURE: Logic did not increase temperature.")
     

--- Logic Verification for P0217 ---
Feature: ecu_7ea_engine_coolant_temperature
Avg Healthy Value:  81.34
Avg Critical Value: 120.48

✅ SUCCESS: Critical temperature is higher than healthy temperature.


In [9]:
path = injector.save_synthetic_data(df_p0217, dtc_code)
print(f"Verified file exists: {path.exists()}")

   -> Saved synthetic dataset: C:\streaming_emulator\DTC\synth_data\engine\synthetic_P0217.csv (Rows: 1000)
Verified file exists: True


In [10]:
import sys
import torch
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path("C:/streaming_emulator")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from DTC.src.fault_injector import FaultInjector
from DTC.src.model_factory import ModelFactory, DTC_Network
from DTC.src import config

print("Imports successful.")

Imports successful.


In [11]:
# 1. Create Data (Using Logic from Step 3)
injector = FaultInjector("engine")
dtc_code = "P0217" # Coolant Overheat
df_train = injector.create_dataset(dtc_code, n_samples=2000)

print(f"Training Data Generated: {len(df_train)} rows")

[ENGINE] Loading reference data source: synthetic_engine_inference_scenarioA_sim001.csv
   -> Generating 2000 samples for P0217 using features: ['ecu_7ea_engine_coolant_temperature', 'engine_oil_temperature', 'engine_load_absolute', 'ecu_7eb_engine_rpm_rpm']
Training Data Generated: 2000 rows


In [12]:
# 2. Initialize Factory and Train
factory = ModelFactory()
dtc_config = config.get_dtc_config("engine", dtc_code)
features = dtc_config['features']

print(f"Features: {features}")

# Train for 100 epochs
model, scaler = factory.train(df_train, features, epochs=100, verbose=True)

Features: ['ecu_7ea_engine_coolant_temperature', 'engine_oil_temperature', 'engine_load_absolute', 'ecu_7eb_engine_rpm_rpm']
   -> Training started (Input Dim: 4, Rows: 2000)
      Epoch [20/100], Loss: 0.5558
      Epoch [40/100], Loss: 0.3570
      Epoch [60/100], Loss: 0.3309
      Epoch [80/100], Loss: 0.3232
      Epoch [100/100], Loss: 0.3206
   -> Training complete. Final Loss: 0.3206


In [13]:
# 3. Verify the model actually learned something
# We take a row we KNOW is healthy and a row we KNOW is critical from the dataframe
# and ask the model to predict them.

# Pick a healthy row (target = 0)
healthy_row = df_train[df_train['target'] == 0.0].iloc[0]
# Pick a critical row (target > 0.9)
critical_row = df_train[df_train['target'] > 0.9].iloc[0]

# Prepare Inputs
X_healthy = scaler.transform([healthy_row[features].values])
X_critical = scaler.transform([critical_row[features].values])

# Predict (Switch to Eval mode)
model.eval()
with torch.no_grad():
    pred_healthy = model(torch.tensor(X_healthy, dtype=torch.float32)).item()
    pred_critical = model(torch.tensor(X_critical, dtype=torch.float32)).item()

print(f"\n--- Model Verification ---")
print(f"Healthy Input Prediction (Should be near 0): {pred_healthy:.4f}")
print(f"Critical Input Prediction (Should be near 1): {pred_critical:.4f}")

if pred_healthy < 0.2 and pred_critical > 0.8:
    print("✅ SUCCESS: Model correctly identifies healthy vs faulty data.")
else:
    print("❌ FAILURE: Model predictions are ambiguous.")


--- Model Verification ---
Healthy Input Prediction (Should be near 0): 0.0222
Critical Input Prediction (Should be near 1): 0.9442
✅ SUCCESS: Model correctly identifies healthy vs faulty data.


In [14]:

# 4. Verify Saving
save_path = factory.save_artifacts(model, scaler, "engine", dtc_code)
print(f"Artifacts exist? {(save_path / 'model.pt').exists()}")

   -> Artifacts saved to: C:\streaming_emulator\DTC\artifacts\engine\P0217
Artifacts exist? True
